# Part A Question 5: Alternative Tokenization Approaches

This notebook compares alternative tokenizers for the Q1 tokenization task: spaCy, TreebankWord, BERT/WordPiece, and Stanza.

In [ ]:
import re
from dataclasses import dataclass
from pathlib import Path
from pprint import pprint
from typing import cast

import pandas as pd
import spacy
import stanza
from nltk.tokenize import TreebankWordTokenizer
from stanza.models.common.doc import Document
from stanza.pipeline.core import DownloadMethod

In [ ]:
DATA_PATH = Path('data/Data_1.txt')
if not DATA_PATH.exists():
    DATA_PATH = Path('../data/Data_1.txt')

text = DATA_PATH.read_text(encoding='utf-8').replace('\r\n', '\n').replace('\n', ' ')
text

## 1. spaCy Tokenizer

In [ ]:
spacy_nlp = spacy.load('en_core_web_sm')
spacy_tokens = [token.text for token in spacy_nlp(text)]
print('Token count:', len(spacy_tokens))
pprint(spacy_tokens)

## 2. TreebankWord Tokenizer

In [ ]:
treebank_tokenizer = TreebankWordTokenizer()
treebank_tokens: list[str] = []
for chunk in re.split(r'(?<=[.!?])\s+(?=[A-Z])', text.strip()):
    treebank_tokens.extend(treebank_tokenizer.tokenize(chunk))

print('Token count:', len(treebank_tokens))
pprint(treebank_tokens)


## 3. BERT/WordPiece Tokenizer

In [ ]:
WORDPIECE_VOCAB = {
    '[UNK]', 'classification', 'task', 'choosing', 'correct', 'class', 'label', 'given',
    'input', 'is', 'basic', 'tasks', 'each', 'instance', 'may', 'be', 'assigned',
    'multiple', 'considered', 'isolation', 'from', 'all', 'other', 'inputs', 'and',
    'the', 'set', 'of', 'labels', 'defined', 'advance', 'has', 'a', 'number',
    'interesting', 'variants', 'for', 'example', 'in', 'multi', '##class', 'open',
    '-', 'not', 'sequence', 'list', 'are', 'jointly', 'classified', '.', ',', ';'
}


def wordpiece_tokenize_word(word: str, vocab: set[str]) -> list[str]:
    if word in {'.', ',', ';'}:
        return [word]

    pieces: list[str] = []
    start = 0
    lowered = word.casefold()

    while start < len(lowered):
        end = len(lowered)
        match = None

        while start < end:
            piece = lowered[start:end]
            candidate = piece if start == 0 else f'##{piece}'
            if candidate in vocab:
                match = candidate
                break
            end -= 1

        if match is None:
            return ['[UNK]']

        pieces.append(match)
        start = end

    return pieces


wordpiece_tokens: list[str] = []
for token in re.findall(r'[A-Za-z]+|[0-9]+|[^\w\s]', text):
    wordpiece_tokens.extend(wordpiece_tokenize_word(token, WORDPIECE_VOCAB))

print('Token count:', len(wordpiece_tokens))
pprint(wordpiece_tokens)

## 4. Stanza Tokenizer

In [ ]:
stanza_pipeline = stanza.Pipeline(
    lang='en',
    processors='tokenize',
    tokenize_no_ssplit=True,
    download_method=DownloadMethod.NONE,
    verbose=False,
)
stanza_document = cast(Document, stanza_pipeline(text))
stanza_tokens = [token.text for sentence in stanza_document.sentences for token in sentence.tokens]
print('Token count:', len(stanza_tokens))
pprint(stanza_tokens)

## 5. Token Count Comparison

In [ ]:
token_count_table = pd.DataFrame(
    {
        'Tokenizer': ['spaCy', 'TreebankWord', 'BERT/WordPiece', 'Stanza'],
        'Token Count': [len(spacy_tokens), len(treebank_tokens), len(wordpiece_tokens), len(stanza_tokens)],
    }
)
token_count_table